<br>
<a href="https://github.com/aperture-systems-lab">
    <img src="assets/banner_semillero.png" width="955" style="margin: 0px 0px 12px;"/>
</a>
<h1 style="line-height: 1.4;"><font color="#29c4d9"><b>De Prompts a Agentes:</b></font> Inteligencia Artificial Agéntica con Python</h1>
<h2><b>Notebook 2: </b>System prompt, caché y herramientas</h2>
<br>


In [1]:
from dotenv import load_dotenv

load_dotenv()

True

In [2]:
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from langchain.messages import HumanMessage

modelo = init_chat_model("google_genai:gemini-3.6-flash")

----

<br>

## **Parte 1:** El system prompt

In [3]:
agente = create_agent(modelo)

respuesta = agente.invoke({"messages": [HumanMessage("¿Puedo vivir con mi gato en el apartamento?")]})
print(respuesta["messages"][-1].text)

La respuesta corta es: **depende principalmente de si eres inquilino o propietario, de tu contrato de alquiler y de las leyes de tu país.**

Para saber si puedes vivir con tu gato en tu apartamento, analiza los siguientes casos:

---

### 1. Si estás ALQUILANDO (Inquilino)
* **Revisa tu contrato de alquiler:** Es el documento más importante.
  * **Si el contrato prohíbe mascotas explícitamente:** Legalmente el propietario puede rescindir el contrato o negarse a alquilarte si llevas un gato.
  * **Si el contrato NO dice nada sobre mascotas:** En la mayoría de los países, lo que no está prohibido en el contrato está permitido. Sin embargo, lo ideal es avisar al propietario para evitar conflictos.
  * **Si vas a firmar un contrato nuevo:** Puedes negociar con el dueño. Ofrecer un depósito adicional (fianza por daños) o presentar la cartilla veterinaria al día (vacunas, esterilización) suele dar confianza.

### 2. Si el apartamento es TUYO (Propietario)
* En la gran mayoría de los casos **

In [ ]:
PROMPT = """
    Eres asesor de la Inmobiliaria Señor Gato, en Medellín.
    - Somos amigos de los gatos: se permiten hasta dos mascotas por inmueble.
    - Responde en dos frases.
    """

agente = create_agent(modelo, system_prompt=PROMPT)

respuesta = agente.invoke({"messages": [HumanMessage("¿Puedo vivir con mi gato en el apartamento?")]})
print(respuesta["messages"][-1].text)

¡Claro que sí, en Inmobiliaria Señor Gato amamos a los felinos! Puedes mudarte con él sin ningún problema, ya que en nuestros inmuebles en Medellín permitimos hasta dos mascotas por apartamento.


----

<br>

## **Parte 2:** El system prompt y la caché

In [5]:
from pathlib import Path

inventario = Path("datos/inmobiliaria.sql").read_text(encoding="utf-8")

SISTEMA = f"""Eres asesor de la Inmobiliaria Señor Gato.
    - Responde en una frase.

INVENTARIO:
{inventario}"""

print(len(SISTEMA), "caracteres")

10410 caracteres


El system prompt se guarda **una vez** en la caché y cada pregunta lo reutiliza.

In [6]:
from google import genai
from google.genai import types

cliente = genai.Client(http_options={"api_version": "v1beta"})
cache = cliente.caches.create(
    model="gemini-3.6-flash",
    config=types.CreateCachedContentConfig(system_instruction=SISTEMA, ttl="600s"),
)

con_cache = init_chat_model("google_genai:gemini-3.6-flash", cached_content=cache.name, api_version="v1beta")

for pregunta in ["¿Cobran depósito?", "¿Cuántos gatos puedo tener?", "¿Cuál es el apartamento más barato?"]:
    respuesta = con_cache.invoke([HumanMessage(pregunta)])
    uso = respuesta.usage_metadata
    print("tokens:", uso["input_tokens"], "  de la caché:", uso["input_token_details"]["cache_read"], "\n")

tokens: 4037   de la caché: 4031 



tokens: 4039   de la caché: 4031 



tokens: 4040   de la caché: 4031 



----

<br>

## **Parte 3:** Tools


<img src="assets/diagramas/bucle.png" width="760"/>

In [7]:
from langchain.tools import tool


@tool
def precio_m2(precio: int, area_m2: float) -> str:
    """Calcula el precio por metro cuadrado de un inmueble, en pesos."""
    return f"${precio / area_m2:,.0f} por m²"


precio_m2.invoke({"precio": 380_000_000, "area_m2": 92})

'$4,130,435 por m²'

In [8]:
print(precio_m2.name)
print(precio_m2.description)
print(precio_m2.args)

precio_m2
Calcula el precio por metro cuadrado de un inmueble, en pesos.
{'precio': {'title': 'Precio', 'type': 'integer'}, 'area_m2': {'title': 'Area M2', 'type': 'number'}}


In [9]:
APARTAMENTOS = {
    "SG-101": {"barrio": "Laureles", "precio": 380_000_000, "area_m2": 92},
    "SG-102": {"barrio": "Laureles", "precio": 415_000_000, "area_m2": 98},
    "SG-401": {"barrio": "Belén", "precio": 295_000_000, "area_m2": 72},
}


@tool
def ver_apartamento(codigo: str) -> dict | str:
    """Devuelve el barrio, el precio en pesos y el área de un apartamento por su código, por ejemplo SG-101."""
    return APARTAMENTOS.get(codigo.upper(), "No existe ese código")

In [10]:
PROMPT = """Eres asesor de la Inmobiliaria Señor Gato.
    - Usa tus herramientas para los datos y las cuentas."""

agente = create_agent(modelo, tools=[ver_apartamento, precio_m2], system_prompt=PROMPT)

respuesta = agente.invoke({"messages": [HumanMessage("¿Cuál sale más barato por metro cuadrado: el SG-101 o el SG-401?")]})

for mensaje in respuesta["messages"]:
    mensaje.pretty_print()

================================ Human Message =================================

¿Cuál sale más barato por metro cuadrado: el SG-101 o el SG-401?
================================== Ai Message ==================================

[]
Tool Calls:
  ver_apartamento (call_319210007669411375)
 Call ID: call_319210007669411375
  Args:
    codigo: SG-101
  ver_apartamento (call_319210007669411536)
 Call ID: call_319210007669411536
  Args:
    codigo: SG-401
================================= Tool Message =================================
Name: ver_apartamento

{"barrio": "Laureles", "precio": 380000000, "area_m2": 92}
================================= Tool Message =================================
Name: ver_apartamento

{"barrio": "Belén", "precio": 295000000, "area_m2": 72}
================================== Ai Message ==================================

[]
Tool Calls:
  precio_m2 (call_9050091558825629441)
 Call ID: call_9050091558825629441
  Args:
    precio: 380000000
    area_m2: 92
  prec

Las peticiones de herramienta que hizo el modelo:

In [11]:
for mensaje in respuesta["messages"]:
    for llamada in getattr(mensaje, "tool_calls", []):
        print(llamada["name"], llamada["args"])

ver_apartamento {'codigo': 'SG-101'}
ver_apartamento {'codigo': 'SG-401'}
precio_m2 {'precio': 380000000, 'area_m2': 92}
precio_m2 {'area_m2': 72, 'precio': 295000000}


-----

<br>

### **Siguiente notebook:**

[`03_interacciones.ipynb`](03_interacciones.ipynb)

### <font color="#29c4d9">**Notebook 2 listo.**</font>

<br>

---
<div style="margin-top: 50px;"><center><a href="https://github.com/aperture-systems-lab"><img src="assets/banner_logo.png" width="955"/></a></center></div>